# 3. Validation

Checks that the tables built in notebook 2 are consistent

In [ ]:
import sys
import pandas as pd

sys.path.append("..")
from src.utils import check

DB = "../data/db"

municipality = pd.read_csv(f"{DB}/municipality.csv", dtype={"ine_code": str})
mun_population = pd.read_csv(f"{DB}/mun_population.csv", dtype={"ine_code": str})
mun_crime = pd.read_csv(f"{DB}/mun_crime.csv", dtype={"ine_code": str})
reg_crime = pd.read_csv(f"{DB}/reg_crime.csv", dtype={"ine_code": str})
reg_offences = pd.read_csv(f"{DB}/reg_offences.csv")
reg_victims = pd.read_csv(f"{DB}/reg_victims.csv")
reg_offenders = pd.read_csv(f"{DB}/reg_offenders.csv")
crime_type_mun = pd.read_csv(f"{DB}/crime_type_mun.csv")

## 3.1 Checks

### Do the crime types add up to the published total?

In [ ]:
comparable = crime_type_mun[crime_type_mun.all_years == 1].code.tolist()

wide = mun_crime.pivot_table(index=["ine_code", "year"], columns="crime_code", values="crime_count")
parts = wide[comparable].sum(axis=1)
total = wide["TOTAL"]
diff = (parts - total).abs()

check("municipal crime types add up to the published total", (diff < 0.001).all(), f"{len(diff)} municipality-year combinations checked")

if (diff >= 0.001).any():
    print(wide[diff >= 0.001])

### Is every municipality in the data a real municipality?

In [ ]:
known = set(municipality.ine_code)
for name, table in [("mun_crime", mun_crime), ("mun_population", mun_population)]:
    unknown = set(table.ine_code) - known
    check(f"{name} only refers to known municipalities", not unknown, str(unknown or ""))

### Are there negative counts?

In [ ]:
check(f"mun_crime has no negative values", (mun_crime.crime_count.dropna() >= 0).all())
check(f"mun_population has no negative values", (mun_population.population.dropna() >= 0).all())

## 3.2 Validation

### Are there any duplicates?

In [ ]:
check("mun_crime has unique (ine_code, year, crime_code)", not mun_crime.duplicated(subset=["ine_code","year","crime_code",]).any())
check("reg_crime has unique (year, crime_code)", not reg_crime.duplicated(subset=["year","crime_code",]).any())
check("mun_population has unique (ine_code, year, sex)", not mun_population.duplicated(subset=["ine_code","year","sex"]).any())
check("municipality has unique (ine_code,municipality_name)", not municipality.duplicated(subset=["ine_code","municipality_name"]).any())
check("reg_offences has unique (year,crime_code)", not reg_offences.duplicated(subset=["year","crime_code"]).any())
check("reg_offenders has unique (year,crime_code,age,sex)", not reg_offenders.duplicated(subset=["year","crime_code","age","sex"]).any())
check("reg_victims has unique (year,crime_code,age,sex)", not reg_victims.duplicated(subset=["year","crime_code","age","sex"]).any())

## 3.3 Known problems to keep in mind

Two  things about this data that cannot be fixed by code: 

- **The 2020 crime file.** Detail on cybercrime does not exist for 2020. The
eleven comparable types work for every year, but any analysis specifically
about cybercrime has a gap in 2020.

- **Coverage.** Only 35 to 37 municipalities out of 179 appear in the crime data.


In [ ]:
print(f"crime types available in 2019: {mun_crime[mun_crime.year == 2019].crime_code.nunique()}")
print(f"crime types available in 2020: {mun_crime[mun_crime.year == 2020].crime_code.nunique()}")
print(f"crime types available in 2021: {mun_crime[mun_crime.year == 2021].crime_code.nunique()}")
print(f"crime types available in 2022: {mun_crime[mun_crime.year == 2022].crime_code.nunique()}")
print(f"crime types available in 2023: {mun_crime[mun_crime.year == 2023].crime_code.nunique()}")
print(f"crime types available in 2024: {mun_crime[mun_crime.year == 2024].crime_code.nunique()}")